In [1]:
# Instalación de dependencias básicas
# IMPORTANTE: Diffusers debe instalarse desde source para soporte de Z-Image

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install git+https://github.com/huggingface/diffusers
# !pip install transformers accelerate safetensors sentencepiece
# !pip install huggingface-hub pillow matplotlib

# [OPCIONAL] Instalar Flash Attention para mejor rendimiento
# Solo si tienes GPU compatible (Ampere/Ada/Hopper)
# !pip install flash-attn --no-build-isolation

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-a46b8wjv
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-a46b8wjv
  Resolved https://github.com/huggingface/diffusers to commit 2fa9b93c4d3e5f40713d4489bea12a2adeb4ab9e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.1/516.1 kB 9.3 MB/s eta 0:00:00ta 0:00:01
  Created wheel for diffusers: filename=diffusers-0.39.0.dev0-py3-none-any.whl size=5485283 sha256=e8b3b3e33afe0aa8e83fd372e5a30e0218bf9a31d14ea782d589dfb8a0c1de47
  Stored in directory: /tmp/pip-ephem-wheel-cache-prgziex1/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.6.2
    Uninstalling safetensors-0.6.2:
      Successfully unins

In [2]:
# Verificar instalación
import torch
from PIL import Image
import os
import time
import random

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("✅ Z-Image Pipeline importado correctamente")

import torch
from diffusers import Flux2KleinPipeline
from diffusers.utils import load_image

device = "cuda"
dtype = torch.bfloat16

# TommyJarvis614/FLUX-NSW
# pipe = Flux2KleinPipeline.from_pretrained("TommyJarvis614/FLUX-NSW", torch_dtype=dtype)
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)
pipe.enable_model_cpu_offload()  # save some VRAM by offloading the model to CPU

✅ PyTorch version: 2.8.0+cu126
✅ CUDA disponible: True
✅ GPU: Tesla T4
✅ VRAM Total: 14.56 GB
✅ Z-Image Pipeline importado correctamente


2026-06-02 01:54:57.810929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780365298.041495      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780365298.112183      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780365298.660291      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780365298.660333      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780365298.660336      57 computation_placer.cc:177] computation placer alr

model_index.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/821 [00:00<?, ?B/s]

transformer/diffusion_pytorch_model.safe(…):   0%|          | 0.00/7.75G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
prompt = "Large explosion at night over a dark city skyline, cinematic documentary style, dark blue and orange color palette, dramatic lighting, sharp focus, geopolitical news aesthetic, no text, no human faces, 4K"

In [3]:
prompt = "Una persona sonriente en un parque"

In [5]:
seeds = [0]
for i in range(3):
    seeds.append(random.randint(10000,99999))

In [ ]:
for seed in seeds:
    result = pipe(
        prompt=prompt,
        # negative_prompt=negative_prompt,
        # image=[face_image], # body_image], #, font_image],
        height=720,
        width=1280,
        guidance_scale=1.0,
        num_inference_steps=6,
        generator=torch.Generator(device=device).manual_seed(seed)
    )
    
    images = result.images
    image = images[0]
    image.save(f"flux-klein-{int(time.time())}-{seed}.png")

  0%|          | 0/6 [00:00<?, ?it/s]